In [ ]:
from torchvision import models
from pathlib import Path
import sys

In [ ]:
NB_DIR = Path.cwd()
PROJECT_ROOT = NB_DIR.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [ ]:
res_net = models.resnet18(weights="DEFAULT")

In [ ]:
res_net

In [ ]:
res_net.fc

##### Check the custom model class

In [ ]:
from src.model import ASLCNNModel

In [ ]:
model = ASLCNNModel()

In [ ]:
model 

##### Test the loaders

In [ ]:
from src.loaders import get_loaders

In [ ]:
test_train_loader, test_val_loader, test_classes = get_loaders(batch_Size=32)

In [ ]:
print(test_classes)

In [ ]:
sample = next(iter(test_train_loader))
image, label = sample
print(image.shape)
print(label.shape)

In [ ]:
sample = next(iter(test_val_loader))
image, label = sample
print(image.shape)
print(label.shape)

##### The dataloaders fetches the dataset properly.

#### Now we train the model

In [ ]:
from src.train import train
import matplotlib.pyplot as plt 

In [ ]:
import torch

In [ ]:
train_history, val_history = train()

In [ ]:
plt.figure(figsize=(12,8))
plt.title("Training and Validation Loss")
plt.plot(train_history, label="Train Loss (Cross Entropy Loss)", marker='o')
plt.plot(val_history, label="Validation Loss (Cross Entropy Loss)", marker='x')
plt.xlabel("Epochs")
plt.ylabel("Cross Entropy Loss")
plt.legend()
plt.show()

In [ ]:
last_train_loss = train_history[-1]
last_val_loss = val_history[-1]
print(f"Last Training Loss: {last_train_loss}")
print(f"Last Validation Loss: {last_val_loss}")

##### Now lets test the model

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import torch
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
import cv2

In [ ]:
test_model = ASLCNNModel()
checkpoint = torch.load(PROJECT_ROOT / "models" / "checkpoints" / "asl_cnn_model.pth")
test_model.load_state_dict(checkpoint['model_state_dict'])

##### Load the test_data

In [ ]:
from src.dataset import ASLDataset
from src.transforms import get_transform


In [ ]:
test_transform = get_transform()['val']

In [ ]:
test_dataset_path = PROJECT_ROOT / "data" / "asl_alphabet_test" / "asl_alphabet_test"

In [ ]:
test_data = ASLDataset()
classes = test_data.classes
print(classes)

In [ ]:
from src.predict import ASLClassifier
classifier = ASLClassifier()

In [ ]:
y_true = []
y_pred = []
image_files = list(test_dataset_path.glob("*.jpg"))
for img_path in tqdm(image_files):
    file_name = img_path.stem
    true_label = file_name.replace("_test", "")
    
    if true_label not in classes:
        print("Skipping unknown label: ", true_label)
        continue
    
    image = cv2.imread(str(img_path))
    if image is None:
        continue
    
    pred_label, confidence = classifier.classify_frame(image)
    y_true.append(true_label)
    y_pred.append(pred_label)

In [ ]:
print("\nClassification Report:")
print(classification_report(y_true, y_pred, labels=classes, target_names=classes, ))

plt.figure(figsize=(20, 15))
cm = confusion_matrix(y_true, y_pred, labels=classes)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=classifier.classes, 
            yticklabels=classifier.classes)

plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()